# Phase 5 — LFA-Specific Optimization

After Phases 3 (design) and 4 (specificity), this phase evaluates whether
candidates will actually work in a lateral flow assay. A nanobody can bind
its target tightly and specifically but still fail in an LFA if:

- Its binding pocket is too enclosed (slow on-rate → weak signal)
- Its paratope faces the gold nanoparticle surface (steric occlusion)
- It unfolds during storage (shelf life failure)

**Three assessments:**
1. Kinetic accessibility — pocket openness, CDR rigidity, charge steering
2. Conjugation orientation — paratope vs C-terminal tag geometry
3. Thermal stability — predicted Tm and Tagg

In [ ]:
import logging
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from omegaconf import OmegaConf

from nanolfa.lfa.stability import predict_stability
from nanolfa.utils.sequence import load_bundled_germlines

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

OUTPUT_DIR = Path('../data/results/lfa_optimization')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load scaffolds as stand-in candidates
scaffolds = load_bundled_germlines()
print(f'Loaded {len(scaffolds)} scaffolds for LFA assessment')

## 1. Thermal Stability Assessment

LFA nanobodies need Tm > 65°C (survive storage at 25°C with margin)
and Tagg > 55°C (no aggregation during gold conjugation at 40°C).

In [ ]:
stability_results = []
for s in scaffolds:
    result = predict_stability(s.sequence, candidate_id=s.name)
    stability_results.append(result)

print(f'{"Name":<30} {"Tm":>6} {"Tagg":>6} {"Score":>6} {"Pass":>6} {"Warnings"}')
print('-' * 90)
for r in stability_results:
    warn = '; '.join(r.warnings[:2]) if r.warnings else 'none'
    print(
        f'{r.candidate_id:<30} {r.predicted_tm:>5.0f}C {r.predicted_tagg:>5.0f}C '
        f'{r.sequence_stability_score:>6.2f} {"PASS" if r.passes_lfa_requirements else "FAIL":>6} '
        f'{warn}'
    )

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Tm distribution
ax = axes[0]
tms = [r.predicted_tm for r in stability_results]
colors = ['#4CAF50' if r.passes_lfa_requirements else '#F44336' for r in stability_results]
ax.barh([r.candidate_id for r in stability_results], tms, color=colors, height=0.6)
ax.axvline(x=65, color='red', linestyle='--', alpha=0.7, label='Min Tm (65°C)')
ax.set_xlabel('Predicted Tm (°C)', size=12)
ax.set_title('Thermal Stability', size=13)
ax.legend()
ax.invert_yaxis()

# Stability score breakdown
ax = axes[1]
names = [r.candidate_id for r in stability_results]
scores = [r.sequence_stability_score for r in stability_results]
flex_penalties = [r.cdr3_flexibility_penalty for r in stability_results]

x = np.arange(len(names))
ax.barh(x, scores, height=0.4, label='Stability score', color='#2196F3', alpha=0.8)
ax.barh(x + 0.4, flex_penalties, height=0.4, label='CDR3 flexibility', color='#FF9800', alpha=0.8)
ax.set_yticks(x + 0.2)
ax.set_yticklabels(names, size=8)
ax.set_xlabel('Score (0-1)', size=12)
ax.set_title('Stability Components', size=13)
ax.legend(fontsize=9)
ax.invert_yaxis()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'thermal_stability.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Conjugation Orientation

When conjugated to a gold nanoparticle via a C-terminal His6 tag,
the paratope (CDR loops) must face **away** from the NP surface.

The orientation check requires a predicted 3D structure. Without
AlphaFold predictions available, we describe the concept and show
what the assessment output looks like.

In [ ]:
# Simulated orientation data
rng = np.random.default_rng(42)

sim_orientations = []
for s in scaffolds:
    angle = rng.normal(120, 25)  # most VHH have favorable geometry
    clearance = rng.normal(35, 10)
    sim_orientations.append({
        'name': s.name,
        'angle': float(np.clip(angle, 40, 180)),
        'clearance': float(np.clip(clearance, 5, 60)),
        'compatible': angle > 90 and clearance > 10,
    })

fig, ax = plt.subplots(figsize=(8, 6))
angles = [o['angle'] for o in sim_orientations]
clearances = [o['clearance'] for o in sim_orientations]
colors = ['#4CAF50' if o['compatible'] else '#F44336' for o in sim_orientations]

ax.scatter(angles, clearances, c=colors, s=100, edgecolors='white', zorder=3)
for o in sim_orientations:
    ax.annotate(o['name'][:15], (o['angle'], o['clearance']), fontsize=7, alpha=0.7)

ax.axvline(x=90, color='red', linestyle='--', alpha=0.5, label='Min angle (90°)')
ax.axhline(y=10, color='orange', linestyle='--', alpha=0.5, label='Min clearance (10 Å)')

# Shade pass zone
ax.axvspan(90, 180, alpha=0.05, color='green')
ax.axhspan(10, 70, alpha=0.05, color='green')

ax.set_xlabel('Paratope-NP Angle (degrees)', size=12)
ax.set_ylabel('Minimum Clearance (Å)', size=12)
ax.set_title('Conjugation Orientation Assessment\n(green = compatible, red = blocked)', size=13)
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'conjugation_orientation.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Kinetic Accessibility

The analyte flows past the LFA test line in ~60 seconds. Nanobodies
with enclosed binding pockets or floppy CDR loops may not bind fast
enough to generate detectable signal.

Three factors determine kinetic accessibility:
- **Pocket openness**: wider entrance → faster diffusion-in
- **CDR rigidity**: rigid loops → lock-and-key (fast kon)
- **Charge complementarity**: electrostatic steering accelerates approach

In [ ]:
# Simulated kinetic profiles
sim_kinetics = []
for s in scaffolds:
    pocket = float(np.clip(rng.normal(0.55, 0.12), 0.2, 0.85))
    rigidity = float(np.clip(rng.normal(0.65, 0.1), 0.3, 0.95))
    charge = float(np.clip(rng.normal(0.5, 0.15), 0.1, 0.9))
    kon = 0.50 * pocket + 0.35 * rigidity + 0.15 * charge
    binding_frac = float(1 - np.exp(-kon * 60 / 30))
    
    sim_kinetics.append({
        'name': s.name,
        'pocket': pocket,
        'rigidity': rigidity,
        'charge': charge,
        'kon': kon,
        'binding_fraction': binding_frac,
        'suitable': kon >= 0.4,
    })

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Stacked component bar chart
ax = axes[0]
names = [k['name'] for k in sim_kinetics]
pockets = [k['pocket'] * 0.50 for k in sim_kinetics]
rigidities = [k['rigidity'] * 0.35 for k in sim_kinetics]
charges = [k['charge'] * 0.15 for k in sim_kinetics]

ax.barh(names, pockets, color='#2196F3', label='Pocket openness (50%)', height=0.6)
ax.barh(names, rigidities, left=pockets, color='#4CAF50', label='CDR rigidity (35%)', height=0.6)
ax.barh(names, charges, left=[p+r for p,r in zip(pockets, rigidities)],
        color='#FF9800', label='Charge comp. (15%)', height=0.6)
ax.axvline(x=0.4, color='red', linestyle='--', alpha=0.5, label='Min kon threshold')
ax.set_xlabel('Relative kon Score', size=12)
ax.set_title('Kinetic Accessibility Components', size=13)
ax.legend(fontsize=8, loc='lower right')
ax.invert_yaxis()

# Binding fraction at 60s
ax = axes[1]
fractions = [k['binding_fraction'] for k in sim_kinetics]
colors = ['#4CAF50' if k['suitable'] else '#F44336' for k in sim_kinetics]
ax.barh(names, fractions, color=colors, height=0.6)
ax.axvline(x=0.5, color='red', linestyle='--', alpha=0.5, label='50% binding target')
ax.set_xlabel('Expected Binding Fraction at 60s', size=12)
ax.set_title('Predicted LFA Signal (60s flow time)', size=13)
ax.legend()
ax.set_xlim(0, 1)
ax.invert_yaxis()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'kinetic_accessibility.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Combined LFA Compatibility

A candidate must pass all three checks to be LFA-compatible.
The composite LFA score combines all three into a single ranking metric.

In [ ]:
# Combine all assessments
combined = []
for stab, orient, kin in zip(stability_results, sim_orientations, sim_kinetics):
    passes = (
        stab.passes_lfa_requirements
        and orient['compatible']
        and kin['suitable']
    )
    lfa_score = (
        stab.sequence_stability_score * 0.33
        + (1.0 if orient['compatible'] else 0.3) * 0.33
        + kin['kon'] * 0.34
    )
    combined.append({
        'name': stab.candidate_id,
        'stability': stab.passes_lfa_requirements,
        'orientation': orient['compatible'],
        'kinetics': kin['suitable'],
        'passes_all': passes,
        'lfa_score': lfa_score,
        'tm': stab.predicted_tm,
    })

passing = [c for c in combined if c['passes_all']]
print(f'LFA-compatible: {len(passing)}/{len(combined)}')
print(f'\n{"Name":<30} {"Stab":>5} {"Orient":>7} {"Kinet":>6} {"LFA":>5} {"Score":>6}')
print('-' * 65)
for c in sorted(combined, key=lambda x: x['lfa_score'], reverse=True):
    s = 'PASS' if c['stability'] else 'FAIL'
    o = 'PASS' if c['orientation'] else 'FAIL'
    k = 'PASS' if c['kinetics'] else 'FAIL'
    overall = 'PASS' if c['passes_all'] else 'FAIL'
    print(f'{c["name"]:<30} {s:>5} {o:>7} {k:>6} {overall:>5} {c["lfa_score"]:>6.2f}')

## Summary

Candidates passing Phase 5 are ready for **experimental validation**.
These are the top 10–20 nanobodies that:

1. Bind the target with high predicted affinity (Phase 3)
2. Discriminate the target from structural analogs (Phase 4)
3. Are kinetically accessible, properly oriented, and thermally stable (Phase 5)

**Next steps:**
- Synthesize as recombinant proteins (E. coli periplasm or Pichia)
- Characterize by SPR/BLI for kinetics (kon, koff, KD)
- Thermal shift assay for Tm/Tagg
- Prototype LFA with gold conjugates
- Feed experimental data back into Phase 6 (scoring recalibration)

---

**Next:** Phase 6 — Experimental Feedback (notebook `06_experimental_feedback.ipynb`)